
# Red-Black Tree Algorithm Visualization

This notebook uses **`diagramSequence`** from `spytial` to step through a **Red-Black Tree** — insertions, fixups (recoloring + rotations), and deletions — frame by frame in a single interactive viewer.

Each call to `insert` or `delete` automatically records snapshots at every algorithmic milestone: BST placement, each fixup case, and the final root-recolor. Use the **Previous / Next** buttons in the viewer to walk through the full narrative.


In [1]:

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from spytial import diagram, diagramSequence
from spytial.annotations import (
    orientation, atomColor, attribute, hideAtom, hideField,
)



## RBNode — annotated for spatial layout

sPyTial decorators on the class handle all visual concerns:
- **left** → below-left, **right** → below-right
- Atom colour matches the node's RED/BLACK colour
- NIL sentinels, primitives, and parent back-pointers are hidden


In [2]:
RED = "red"
BLACK = "black"

# ── Spatial layout: left child below-left, right child below-right ──
@orientation(selector='left & (RBNode -> (RBNode - NoneType.~key))', directions=['below', 'left'])
@orientation(selector='right & (RBNode -> (RBNode - NoneType.~key))', directions=['below', 'right'])
# ── Visual convention: colour the atom background to match the node colour ──
@atomColor(
    selector='{ x : RBNode | @:(x.color) = red }',
    value='red'
)
@atomColor(
    selector='{ x : RBNode | @:(x.color) = black }',
    value='black'
)
# ── Show the key value and colour label inside each atom ──
@attribute(field="key")
@attribute(field="color")
# ── Hide NIL sentinels, the tree wrapper, raw ints, NoneType, and colour strings ──
@hideAtom(
    selector=(
        '{ x : RBNode | (x.key in NoneType) } '
        '+ int + NoneType '
        '+ {s : str | @:s = red or @:s = black}'
    )
)
@hideField(field='parent')  # back-pointers clutter the layout
class RBNode:
    """A single node in a Red-Black Tree."""

    def __init__(self, key=None, color=BLACK, left=None, right=None, parent=None):
        self.key = key
        self.color = color
        self.left = left
        self.right = right
        self.parent = parent

    def __repr__(self):
        if self.key is None:
            return "()"
        lstr = "()" if self.left  is NIL or self.left.key  is None else repr(self.left)
        rstr = "()" if self.right is NIL or self.right.key is None else repr(self.right)
        return f"RBNode(key={self.key}, color={self.color!r}, left={lstr}, right={rstr})"


# ── Singleton NIL sentinel — all leaves point here ──
NIL = RBNode(key=None, color=BLACK)
NIL.left = NIL.right = NIL.parent = NIL

print("RBNode defined with NIL sentinel ✓")


RBNode defined with NIL sentinel ✓



## RBTree — CLRS implementation with built-in snapshots

Every fixup case records a snapshot with a human-readable label. The `identity` function maps each node to `"node-{key}"` so the `stability` policy keeps atoms anchored across frames.


In [3]:

class RBTree:
    """CLRS Red-Black Tree with automatic snapshot recording."""

    def __init__(self):
        self.root = NIL
        self.step = "Initial (empty)"
        self.snapshots: list = []

    # ── Snapshot helpers ───────────────────────────────────────────────────────

    @staticmethod
    def _copy_subtree(node):
        if node is NIL:
            return NIL
        n = RBNode(key=node.key, color=node.color, left=NIL, right=NIL, parent=NIL)
        n.left  = RBTree._copy_subtree(node.left)
        n.right = RBTree._copy_subtree(node.right)
        if n.left  is not NIL: n.left.parent  = n
        if n.right is not NIL: n.right.parent = n
        return n

    def _snap(self, label: str):
        snap = RBTree.__new__(RBTree)
        snap.root      = RBTree._copy_subtree(self.root)
        snap.step      = label
        snap.snapshots = []
        self.snapshots.append(snap)

    # ── Read-only helpers ──────────────────────────────────────────────────────

    def search(self, key):
        x = self.root
        while x is not NIL and key != x.key:
            x = x.left if key < x.key else x.right
        return x

    def minimum(self, x):
        while x.left is not NIL:
            x = x.left
        return x

    def inorder_keys(self, node=None, acc=None):
        if node is None:
            node, acc = self.root, []
        if node is NIL:
            return acc
        self.inorder_keys(node.left, acc)
        acc.append(node.key)
        self.inorder_keys(node.right, acc)
        return acc

    def __repr__(self):
        return f"RBTree(keys={self.inorder_keys()}, step={self.step!r})"

    # ── Rotations (CLRS §13.2) ─────────────────────────────────────────────────

    def left_rotate(self, x):
        y = x.right; x.right = y.left
        if y.left is not NIL: y.left.parent = x
        y.parent = x.parent
        if   x.parent is NIL:      self.root = y
        elif x is x.parent.left:   x.parent.left = y
        else:                       x.parent.right = y
        y.left = x; x.parent = y

    def right_rotate(self, y):
        x = y.left; y.left = x.right
        if x.right is not NIL: x.right.parent = y
        x.parent = y.parent
        if   y.parent is NIL:      self.root = x
        elif y is y.parent.left:   y.parent.left = x
        else:                       y.parent.right = x
        x.right = y; y.parent = x

    # ── Insert + fixup (CLRS §13.3) ────────────────────────────────────────────

    def insert(self, key):
        z = RBNode(key=key, color=RED, left=NIL, right=NIL, parent=None)
        y, x = NIL, self.root
        while x is not NIL:
            y = x; x = x.left if z.key < x.key else x.right
        z.parent = y
        if   y is NIL:        self.root = z
        elif z.key < y.key:   y.left = z
        else:                  y.right = z
        self._snap(f"BST insert: {key} (color=RED)")
        self._insert_fixup(z)

    def _insert_fixup(self, z):
        while z.parent.color is RED:
            if z.parent is z.parent.parent.left:
                uncle = z.parent.parent.right
                if uncle.color is RED:
                    z.parent.color = BLACK; uncle.color = BLACK
                    z.parent.parent.color = RED; z = z.parent.parent
                    self._snap("Insert fixup Case 1: recolor (uncle RED)")
                else:
                    if z is z.parent.right:
                        z = z.parent; self.left_rotate(z)
                        self._snap("Insert fixup Case 2: left-rotate (zig-zag)")
                    z.parent.color = BLACK; z.parent.parent.color = RED
                    self.right_rotate(z.parent.parent)
                    self._snap("Insert fixup Case 3: right-rotate + recolor")
            else:
                uncle = z.parent.parent.left
                if uncle.color is RED:
                    z.parent.color = BLACK; uncle.color = BLACK
                    z.parent.parent.color = RED; z = z.parent.parent
                    self._snap("Insert fixup Case 1 (mirror): recolor")
                else:
                    if z is z.parent.left:
                        z = z.parent; self.right_rotate(z)
                        self._snap("Insert fixup Case 2 (mirror): right-rotate")
                    z.parent.color = BLACK; z.parent.parent.color = RED
                    self.left_rotate(z.parent.parent)
                    self._snap("Insert fixup Case 3 (mirror): left-rotate + recolor")
        self.root.color = BLACK
        self._snap("Insert done: root → BLACK")

    # ── Transplant (CLRS §13.4) ────────────────────────────────────────────────

    def _transplant(self, u, v):
        if   u.parent is NIL:      self.root = v
        elif u is u.parent.left:   u.parent.left = v
        else:                       u.parent.right = v
        v.parent = u.parent

    # ── Delete + fixup (CLRS §13.4) ────────────────────────────────────────────

    def delete(self, key):
        z = self.search(key)
        if z is NIL: return
        self._snap(f"Delete: target node {key}")
        y, y_orig = z, z.color
        if z.left is NIL:
            x = z.right; self._transplant(z, z.right)
        elif z.right is NIL:
            x = z.left; self._transplant(z, z.left)
        else:
            y = self.minimum(z.right); y_orig = y.color; x = y.right
            if y.parent is z: x.parent = y
            else:
                self._transplant(y, y.right); y.right = z.right; y.right.parent = y
            self._transplant(z, y); y.left = z.left; y.left.parent = y; y.color = z.color
        self._snap(f"Delete: node {key} spliced out")
        if y_orig is BLACK: self._delete_fixup(x)
        else:               self._snap("Delete: no fixup (removed RED)")

    def _delete_fixup(self, x):
        it = 0
        while x is not self.root and x.color is BLACK:
            it += 1
            if x is x.parent.left:
                w = x.parent.right
                if w.color is RED:
                    w.color = BLACK; x.parent.color = RED
                    self.left_rotate(x.parent); w = x.parent.right
                    self._snap(f"Delete fixup Case 1: sibling RED → rotate (iter {it})")
                if w.left.color is BLACK and w.right.color is BLACK:
                    w.color = RED; x = x.parent
                    self._snap(f"Delete fixup Case 2: both nephews BLACK (iter {it})")
                else:
                    if w.right.color is BLACK:
                        w.left.color = BLACK; w.color = RED
                        self.right_rotate(w); w = x.parent.right
                        self._snap(f"Delete fixup Case 3: near nephew RED (iter {it})")
                    w.color = x.parent.color; x.parent.color = BLACK; w.right.color = BLACK
                    self.left_rotate(x.parent)
                    self._snap(f"Delete fixup Case 4: far nephew RED → done (iter {it})")
                    x = self.root
            else:
                w = x.parent.left
                if w.color is RED:
                    w.color = BLACK; x.parent.color = RED
                    self.right_rotate(x.parent); w = x.parent.left
                    self._snap(f"Delete fixup Case 1 (mirror) (iter {it})")
                if w.right.color is BLACK and w.left.color is BLACK:
                    w.color = RED; x = x.parent
                    self._snap(f"Delete fixup Case 2 (mirror) (iter {it})")
                else:
                    if w.left.color is BLACK:
                        w.right.color = BLACK; w.color = RED
                        self.left_rotate(w); w = x.parent.left
                        self._snap(f"Delete fixup Case 3 (mirror) (iter {it})")
                    w.color = x.parent.color; x.parent.color = BLACK; w.left.color = BLACK
                    self.right_rotate(x.parent)
                    self._snap(f"Delete fixup Case 4 (mirror) (iter {it})")
                    x = self.root
        x.color = BLACK
        self._snap("Delete fixup done")


def rbt_identity(obj):
    """Stable cross-frame identity: maps each RBNode to 'node-{key}'."""
    if isinstance(obj, RBNode) and obj.key is not None:
        return f"node-{obj.key}"
    return None

print("RBTree + rbt_identity defined ✓")


RBTree + rbt_identity defined ✓



## Run the algorithm

Insert **[26, 17, 41, 47, 30, 38]**, then delete **17** and **41**.  
Every fixup case — recoloring, left/right rotations — is captured automatically.


In [4]:

tree = RBTree()

# Phase 1: insertions
for k in [26, 17, 41, 47, 30, 38]:
    tree.insert(k)

# Phase 2: deletions
tree.delete(17)
tree.delete(41)

print(f"Final inorder: {tree.inorder_keys()}")
print(f"Total frames:  {len(tree.snapshots)}")
for i, s in enumerate(tree.snapshots):
    print(f"  [{i:02d}] {s.step}")


Final inorder: [26, 30, 38, 47]
Total frames:  24
  [00] BST insert: 26 (color=RED)
  [01] Insert done: root → BLACK
  [02] BST insert: 17 (color=RED)
  [03] Insert done: root → BLACK
  [04] BST insert: 41 (color=RED)
  [05] Insert done: root → BLACK
  [06] BST insert: 47 (color=RED)
  [07] Insert fixup Case 1 (mirror): recolor
  [08] Insert done: root → BLACK
  [09] BST insert: 30 (color=RED)
  [10] Insert done: root → BLACK
  [11] BST insert: 38 (color=RED)
  [12] Insert fixup Case 1: recolor (uncle RED)
  [13] Insert done: root → BLACK
  [14] Delete: target node 17
  [15] Delete: node 17 spliced out
  [16] Delete fixup Case 1: sibling RED → rotate (iter 1)
  [17] Delete fixup Case 4: far nephew RED → done (iter 1)
  [18] Delete fixup done
  [19] Delete: target node 41
  [20] Delete: node 41 spliced out
  [21] Delete fixup Case 1 (mirror) (iter 1)
  [22] Delete fixup Case 2 (mirror) (iter 1)
  [23] Delete fixup done


In [5]:

diagramSequence(
    tree.snapshots,
    sequence_policy="stability",
    identity=rbt_identity,
    as_type=RBTree,
    title="Red-Black Tree: insert [26,17,41,47,30,38] → delete 17,41",
)
